# 00 — preflight

Toolchain and working copies. Everything later notebooks rely on:
the monorepo checkout with installed workspaces, the fixture site
(netsnek.com), the emailwerk checkout and the zitadel-gql facade SDL.


In [ ]:
import jaen_testkit as k
k.start_run('00-preflight')
print(k.CONFIG['repo_root'])

In [ ]:
with k.section('toolchain'):
    with k.check('node >= 18 present') as c:
        r = c.require(k.sh('node --version'))
        major = int(r.text.lstrip('v').split('.')[0])
        c.expect_true(major >= 18, r.text)

    with k.check('yarn classic present') as c:
        r = c.require(k.sh('yarn --version'))
        c.expect_true(r.text.startswith('1.'), r.text)

    with k.check('python xml parser works') as c:
        import xml.etree.ElementTree as ET
        root = ET.fromstring('<a><b/></a>')
        c.expect_equal(len(root), 1)


In [ ]:
import os
with k.section('working copies'):
    with k.check('monorepo present with feature branch') as c:
        r = c.require(k.sh('git rev-parse --abbrev-ref HEAD', cwd=k.CONFIG['repo_root']))
        c.note(r.text)
        c.expect_true(len(r.text) > 0)

    with k.check('workspaces installed (node_modules)') as c:
        c.expect_true(
            os.path.isdir(k.repo_path('node_modules')),
            'node_modules present')

    with k.check('fixture site checkout present') as c:
        if not os.path.isdir(k.CONFIG['site_dir']):
            c.skip('JAEN_SITE_DIR missing: %s' % k.CONFIG['site_dir'])
        c.expect_true(os.path.isfile(k.site_path('gatsby-config.ts')),
                      k.CONFIG['site_dir'])

    with k.check('emailwerk checkout present') as c:
        if not os.path.isdir(k.CONFIG['emailwerk_dir']):
            c.skip('JAEN_EMAILWERK_DIR missing')
        c.expect_true(
            os.path.isfile(os.path.join(k.CONFIG['emailwerk_dir'], 'src', 'index.ts')),
            'src/index.ts found')

    with k.check('zitadel-gql facade SDL present') as c:
        if not os.path.isfile(k.CONFIG['iam_sdl']):
            c.skip('JAEN_IAM_SDL missing: %s' % k.CONFIG['iam_sdl'])
        text = k.read_text(k.CONFIG['iam_sdl'], '')
        c.expect_contains(text, 'type Mutation')


In [ ]:
with k.section('monorepo layout'):
    with k.check('renamed package gatsby-jaen-emailwerk exists') as c:
        c.expect_true(os.path.isdir(k.repo_path('packages/gatsby-jaen-emailwerk')),
                      'packages/gatsby-jaen-emailwerk')
        c.expect_true(not os.path.isdir(k.repo_path('packages/gatsby-jaen-mailpress')),
                      'old mailpress dir gone')

    with k.check('zitadel-gql client exists in jaen core') as c:
        c.expect_true(os.path.isfile(
            k.repo_path('packages/jaen/src/clients/zitadel-gql/index.ts')),
            'client file present')


In [ ]:
k.summary()
k.save_results('results-00-preflight.json')
rc = k.verdict()
assert rc == 0, 'run has FAILures — see the summary above'